# 03 — Treinamento baseline YOLOv8n no Google Colab

Este notebook executa o **primeiro treinamento baseline** do projeto.

## Objetivo

Treinar um detector YOLOv8n utilizando o dataset HomeObjects-3K preparado anteriormente:

- `train`: treinamento do modelo;
- `val`: acompanhamento durante o treinamento;
- `test`: avaliação final independente.

O conjunto de **teste não é utilizado durante o treinamento**.

## Fluxo

```text
Dataset processado
       ↓
Google Drive
       ↓
Google Colab + GPU
       ↓
YOLOv8n pré-treinado
       ↓
Fine-tuning
       ↓
best.pt
       ↓
Avaliação no TEST
       ↓
Métricas + gráficos + pesos
```

---
**Antes de executar este notebook:**

No seu computador, compacte a pasta:

```text
data/processed/homeobjects_yolo
```

gerando:

```text
homeobjects_yolo.zip
```

O ZIP deve estar disponível no Google Drive em:

```text
Meu Drive/
└── TCC_Visao_Computacional/
    └── datasets/
        └── homeobjects_yolo.zip
```

## 1. Ativar GPU no Colab

No Google Colab, acesse:

```text
Ambiente de execução
    → Alterar tipo de ambiente de execução
    → Acelerador de hardware
    → GPU
```

Depois execute a célula abaixo.

In [ ]:
import platform
import sys

import torch

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU não está disponível. "
        "No Colab, ative uma GPU antes de continuar."
    )

## 2. Instalar Ultralytics

A versão instalada será registrada para manter a rastreabilidade do experimento.

In [ ]:
!pip install -q ultralytics

In [ ]:
import ultralytics

from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)

## 3. Montar o Google Drive

O Drive será utilizado para:

- ler o ZIP do dataset;
- salvar pesos do modelo;
- salvar métricas;
- salvar gráficos;
- preservar os resultados mesmo após o encerramento da sessão do Colab.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Configurar caminhos e parâmetros do experimento

Para o primeiro baseline usaremos:

- modelo: `yolov8n.pt`;
- resolução: `640 x 640`;
- épocas: `50`;
- batch automático;
- seed: `42`.

Esses valores poderão ser alterados em experimentos futuros, mas o primeiro baseline deve permanecer registrado.

In [ ]:
from pathlib import Path

# -----------------------------
# Caminhos no Google Drive
# -----------------------------
DRIVE_TCC = Path("/content/drive/MyDrive/TCC_Visao_Computacional")

ZIP_DATASET = DRIVE_TCC / "datasets" / "homeobjects_yolo.zip"
PASTA_RESULTADOS_DRIVE = DRIVE_TCC / "resultados" / "yolov8n_baseline"

# -----------------------------
# Caminhos temporários do Colab
# -----------------------------
PASTA_TRABALHO = Path("/content/tcc_visao_computacional")
PASTA_DATASET = PASTA_TRABALHO / "homeobjects_yolo"

# -----------------------------
# Parâmetros do experimento
# -----------------------------
MODELO_BASE = "yolov8n.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = -1
SEED = 42
PATIENCE = 15
WORKERS = 2

# Treinamento normal
MODO_TESTE_RAPIDO = False

if MODO_TESTE_RAPIDO:
    EPOCHS = 3

PASTA_RESULTADOS_DRIVE.mkdir(parents=True, exist_ok=True)

print("ZIP do dataset:", ZIP_DATASET)
print("Resultados:", PASTA_RESULTADOS_DRIVE)
print()
print("Modelo:", MODELO_BASE)
print("Épocas:", EPOCHS)
print("Imagem:", IMGSZ)
print("Batch:", BATCH)
print("Seed:", SEED)

## 5. Verificar se o dataset está no Drive

Se esta célula gerar erro, confira se o arquivo está exatamente em:

```text
MyDrive/TCC_Visao_Computacional/datasets/homeobjects_yolo.zip
```

In [ ]:
if not ZIP_DATASET.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado em:\n{ZIP_DATASET}\n\n"
        'Arquivo homeobjects_yolo.zip esperado na pasta indicada no Google Drive.'
    )

tamanho_mb = ZIP_DATASET.stat().st_size / (1024 ** 2)

print("Dataset encontrado.")
print(f"Tamanho do ZIP: {tamanho_mb:.2f} MB")

## 6. Extrair o dataset no armazenamento temporário do Colab

O treinamento ocorrerá utilizando o armazenamento local do Colab, que é mais rápido do que ler milhares de imagens diretamente do Google Drive.

In [ ]:
import shutil

if PASTA_TRABALHO.exists():
    shutil.rmtree(PASTA_TRABALHO)

PASTA_TRABALHO.mkdir(parents=True, exist_ok=True)

shutil.unpack_archive(
    filename=str(ZIP_DATASET),
    extract_dir=str(PASTA_TRABALHO),
)

print("ZIP extraído.")

## 7. Localizar automaticamente o `data.yaml`

Essa etapa torna o notebook tolerante ao caso do ZIP conter ou não uma pasta adicional na raiz.

In [ ]:
arquivos_yaml = list(PASTA_TRABALHO.rglob("data.yaml"))

if len(arquivos_yaml) == 0:
    raise FileNotFoundError("Nenhum data.yaml foi encontrado após a extração.")

if len(arquivos_yaml) > 1:
    print("Foram encontrados vários data.yaml:")
    for arquivo in arquivos_yaml:
        print(" -", arquivo)

    raise RuntimeError(
        "Há mais de um data.yaml. "
        "Mantenha apenas o dataset homeobjects_yolo no ZIP."
    )

CAMINHO_YAML_ORIGINAL = arquivos_yaml[0]
RAIZ_DATASET = CAMINHO_YAML_ORIGINAL.parent

print("Dataset localizado em:", RAIZ_DATASET)
print("YAML original:", CAMINHO_YAML_ORIGINAL)

## 8. Ajustar o `data.yaml` para o ambiente Colab

O YAML criado no Windows contém um caminho absoluto do computador local.

Aqui criamos uma cópia específica para o Colab, mantendo:

- `train`;
- `val`;
- `test`;
- as 12 classes.

Apenas o campo `path` é atualizado.

In [ ]:
import yaml

with CAMINHO_YAML_ORIGINAL.open("r", encoding="utf-8") as arquivo:
    dados_yaml = yaml.safe_load(arquivo)

dados_yaml["path"] = RAIZ_DATASET.as_posix()

CAMINHO_YAML_COLAB = RAIZ_DATASET / "data_colab.yaml"

with CAMINHO_YAML_COLAB.open("w", encoding="utf-8") as arquivo:
    yaml.safe_dump(
        dados_yaml,
        arquivo,
        allow_unicode=True,
        sort_keys=False,
    )

print(CAMINHO_YAML_COLAB.read_text(encoding="utf-8"))

## 9. Validar a estrutura do dataset

Antes do treinamento, confirmamos novamente a quantidade de imagens e labels.

O resultado esperado da preparação anterior é aproximadamente:

```text
train: 1822
val:    456
test:   403
```

In [ ]:
EXTENSOES_IMAGEM = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def contar_imagens(pasta: Path) -> int:
    return sum(
        1
        for arquivo in pasta.rglob("*")
        if arquivo.is_file()
        and arquivo.suffix.lower() in EXTENSOES_IMAGEM
    )


resumo_dataset = []

for split in ("train", "val", "test"):
    pasta_imagens = RAIZ_DATASET / "images" / split
    pasta_labels = RAIZ_DATASET / "labels" / split

    qtd_imagens = contar_imagens(pasta_imagens)
    qtd_labels = len(list(pasta_labels.rglob("*.txt")))

    resumo_dataset.append(
        {
            "split": split,
            "imagens": qtd_imagens,
            "labels": qtd_labels,
        }
    )

import pandas as pd

df_resumo = pd.DataFrame(resumo_dataset)
display(df_resumo)

if not (df_resumo["imagens"] == df_resumo["labels"]).all():
    raise RuntimeError(
        "A quantidade de imagens e labels não coincide."
    )

if df_resumo["imagens"].sum() == 0:
    raise RuntimeError("Dataset vazio.")

print("Estrutura validada.")

## 10. Registrar informações do experimento

Salvamos os principais parâmetros antes do treinamento para garantir a rastreabilidade metodológica.

In [ ]:
import json
from datetime import datetime

informacoes_experimento = {
    "data_execucao_utc": datetime.utcnow().isoformat(),
    "modelo_base": MODELO_BASE,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "patience": PATIENCE,
    "workers": WORKERS,
    "pytorch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "dataset": df_resumo.to_dict(orient="records"),
    "classes": dados_yaml["names"],
}

caminho_info = PASTA_RESULTADOS_DRIVE / "configuracao_experimento.json"

with caminho_info.open("w", encoding="utf-8") as arquivo:
    json.dump(
        informacoes_experimento,
        arquivo,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(informacoes_experimento, ensure_ascii=False, indent=2))

# 11. Treinamento baseline

Agora iniciamos o fine-tuning do **YOLOv8n pré-treinado**.

O modelo utilizará somente:

```text
train
   +
validation
```

O conjunto `test` continuará reservado.

In [ ]:
modelo = YOLO(MODELO_BASE)

resultado_treino = modelo.train(
    data=str(CAMINHO_YAML_COLAB),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    workers=WORKERS,
    device=0,
    pretrained=True,
    optimizer="auto",
    plots=True,
    save=True,
    verbose=True,
    project="/content/experimentos",
    name="yolov8n_baseline",
    exist_ok=True,
)

## 12. Localizar o melhor modelo

O Ultralytics salva dois pesos principais:

```text
best.pt
last.pt
```

Para as próximas etapas do TCC utilizaremos principalmente o **`best.pt`**.

In [ ]:
PASTA_RUN = Path("/content/experimentos/yolov8n_baseline")
CAMINHO_BEST = PASTA_RUN / "weights" / "best.pt"
CAMINHO_LAST = PASTA_RUN / "weights" / "last.pt"

if not CAMINHO_BEST.exists():
    raise FileNotFoundError(
        f"O best.pt não foi encontrado em {CAMINHO_BEST}"
    )

print("Best:", CAMINHO_BEST)
print("Last:", CAMINHO_LAST)
print()
print(f"Tamanho best.pt: {CAMINHO_BEST.stat().st_size / 1024**2:.2f} MB")

# 13. Avaliação final no conjunto TEST

Esta é uma etapa importante para o TCC.

O modelo treinado é carregado a partir do `best.pt` e avaliado nas **403 imagens reservadas para teste**.

As principais métricas de interesse serão:

- Precision;
- Recall;
- mAP@50;
- mAP@50-95.

Além disso, o Ultralytics gerará gráficos e matriz de confusão.

In [ ]:
melhor_modelo = YOLO(str(CAMINHO_BEST))

resultado_test = melhor_modelo.val(
    data=str(CAMINHO_YAML_COLAB),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    plots=True,
    verbose=True,
    project="/content/experimentos",
    name="yolov8n_baseline_test",
    exist_ok=True,
)

## 14. Resumo das métricas do TEST

As métricas abaixo serão salvas também em CSV para uso posterior nos capítulos do TCC.

In [ ]:
metricas_test = {
    "precision": float(resultado_test.box.mp),
    "recall": float(resultado_test.box.mr),
    "map50": float(resultado_test.box.map50),
    "map50_95": float(resultado_test.box.map),
}

df_metricas_test = pd.DataFrame(
    [
        {"metrica": metrica, "valor": valor}
        for metrica, valor in metricas_test.items()
    ]
)

display(df_metricas_test)

df_metricas_test.to_csv(
    PASTA_RESULTADOS_DRIVE / "metricas_test.csv",
    index=False,
    encoding="utf-8-sig",
)

## 15. Métricas por classe

Como o HomeObjects-3K possui forte desbalanceamento entre classes, não devemos avaliar o modelo apenas pelo mAP global.

Esta etapa registra o desempenho individual das 12 classes.

In [ ]:
nomes_classes = melhor_modelo.names

metricas_classes = []

for indice, classe_id in enumerate(resultado_test.box.ap_class_index):
    classe_id = int(classe_id)

    metricas_classes.append(
        {
            "id_classe": classe_id,
            "classe": nomes_classes[classe_id],
            "precision": float(resultado_test.box.p[indice]),
            "recall": float(resultado_test.box.r[indice]),
            "map50": float(resultado_test.box.ap50[indice]),
            "map50_95": float(resultado_test.box.ap[indice]),
        }
    )

df_metricas_classes = pd.DataFrame(metricas_classes)

display(
    df_metricas_classes.sort_values(
        "map50_95",
        ascending=False,
    )
)

df_metricas_classes.to_csv(
    PASTA_RESULTADOS_DRIVE / "metricas_por_classe_test.csv",
    index=False,
    encoding="utf-8-sig",
)

## 16. Copiar todos os resultados para o Google Drive

Serão preservados:

- `best.pt`;
- `last.pt`;
- `results.csv`;
- curvas de treinamento;
- matriz de confusão;
- gráficos de Precision/Recall;
- avaliação do TEST;
- configurações do experimento.

In [ ]:
import shutil

DESTINO_TREINO = PASTA_RESULTADOS_DRIVE / "treinamento"
DESTINO_TEST = PASTA_RESULTADOS_DRIVE / "avaliacao_test"

if DESTINO_TREINO.exists():
    shutil.rmtree(DESTINO_TREINO)

if DESTINO_TEST.exists():
    shutil.rmtree(DESTINO_TEST)

shutil.copytree(
    "/content/experimentos/yolov8n_baseline",
    DESTINO_TREINO,
)

shutil.copytree(
    "/content/experimentos/yolov8n_baseline_test",
    DESTINO_TEST,
)

# Cópia direta do principal artefato do projeto.
shutil.copy2(
    CAMINHO_BEST,
    PASTA_RESULTADOS_DRIVE / "best.pt",
)

print("Resultados salvos em:")
print(PASTA_RESULTADOS_DRIVE)

## 17. Testar o modelo em algumas imagens do TEST

Esta etapa gera exemplos visuais das predições do modelo treinado.

Ela **não altera as métricas**. Serve apenas para inspeção qualitativa.

In [ ]:
import random

random.seed(SEED)

pasta_test = RAIZ_DATASET / "images" / "test"

imagens_test = [
    arquivo
    for arquivo in pasta_test.rglob("*")
    if arquivo.is_file()
    and arquivo.suffix.lower() in EXTENSOES_IMAGEM
]

quantidade_amostras = min(8, len(imagens_test))

amostras = random.sample(
    imagens_test,
    quantidade_amostras,
)

resultados_predicao = melhor_modelo.predict(
    source=[str(imagem) for imagem in amostras],
    imgsz=IMGSZ,
    conf=0.25,
    device=0,
    save=True,
    project="/content/experimentos",
    name="amostras_test",
    exist_ok=True,
)

print(f"{quantidade_amostras} imagens processadas.")

## 18. Salvar exemplos de predição no Drive

In [ ]:
DESTINO_AMOSTRAS = PASTA_RESULTADOS_DRIVE / "amostras_test"

if DESTINO_AMOSTRAS.exists():
    shutil.rmtree(DESTINO_AMOSTRAS)

shutil.copytree(
    "/content/experimentos/amostras_test",
    DESTINO_AMOSTRAS,
)

print("Amostras salvas em:")
print(DESTINO_AMOSTRAS)

# 19. Resultado esperado

Após a execução completa, seu Google Drive deverá conter:

```text
TCC_Visao_Computacional/
│
├── datasets/
│   └── homeobjects_yolo.zip
│
└── resultados/
    └── yolov8n_baseline/
        │
        ├── best.pt
        ├── configuracao_experimento.json
        ├── metricas_test.csv
        ├── metricas_por_classe_test.csv
        │
        ├── treinamento/
        │   ├── weights/
        │   │   ├── best.pt
        │   │   └── last.pt
        │   ├── results.csv
        │   └── ...
        │
        ├── avaliacao_test/
        │   ├── confusion_matrix.png
        │   └── ...
        │
        └── amostras_test/
```

## Próxima etapa

Depois deste baseline, teremos dados suficientes para analisar:

1. se o YOLOv8n aprendeu adequadamente as 12 classes;
2. quais classes apresentam maior dificuldade;
3. se o desbalanceamento afetou `bed`, `laptop`, `door` etc.;
4. se devemos ajustar hiperparâmetros ou manter o baseline;
5. iniciar o módulo de **avaliação de risco à locomoção**.

**Não altere o conjunto TEST entre experimentos.**